# DentalAI Pro: Caries (Cavity) Detection Training Pipeline
This notebook contains the complete pipeline to download the dental caries dataset, preprocess the X-ray images, perform data augmentation, and train an **EfficientNetB0** model using transfer learning.

## 1. Setup Kaggle API & Download Dataset
First, make sure you upload your `kaggle.json` credentials to your home directory (`~/.kaggle/kaggle.json`).

In [ ]:
# Install kaggle package
!pip install -q kaggle

import os
import json
from pathlib import Path

# Note: To run this automatically, place your kaggle.json in the current directory or setup environmental variables:
# os.environ['KAGGLE_USERNAME'] = "your_kaggle_username"
# os.environ['KAGGLE_KEY'] = "your_kaggle_key"

# Download the dataset
!kaggle datasets download -d pilot11/dental-caries-detection-dataset --unzip -p ../datasets/caries

## 2. Load Libraries & Setup Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

## 3. Data Preprocessing & Augmentation
Images will be resized to 224x224. We will partition the dataset into 70% training, 15% validation, and 15% testing, applying rotation, zoom, flips, and brightness adjustments to avoid overfitting.

In [ ]:
img_size = (224, 224)
batch_size = 32
data_dir = "../datasets/caries"

# We will use splitfolders to split the dataset or Keras flow_from_directory with split
# For robust division, let's configure standard ImageDataGenerator split:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.3 # 30% for val/test combined
)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

# For validation, we use the remaining 30%
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.5)

val_generator = val_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training', # First half of the 30% split = 15%
    shuffle=False
)

test_generator = val_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation', # Second half of the 30% split = 15%
    shuffle=False
)

## 4. Build EfficientNetB0 Model (Transfer Learning)
We load EfficientNetB0 pre-trained on ImageNet, discard the top dense layer, and add customized GlobalAveragePooling, Dropout, and Dense classification layers.

In [ ]:
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # Freeze base layers first

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

## 5. Training - Phase 1: Frozen Base

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('../backend/models/caries_model.h5', monitor='val_loss', save_best_only=True)
]

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks
)

## 6. Training - Phase 2: Fine-Tuning
Unfreeze the top layers of EfficientNetB0 and train with a smaller learning rate to optimize weights.

In [ ]:
base_model.trainable = True
# Keep early layers frozen, fine-tune the rest
for layer in base_model.layers[:-30]:
    layer.trainable = False
    
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

history_fine = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks
)

## 7. Model Evaluation & Plotting

In [ ]:
# Plot Accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'] + history_fine.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'] + history_fine.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'] + history_fine.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'] + history_fine.history['val_loss'], label='Val Loss')
plt.title('Loss vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Predictions & Reports
Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)

print('Confusion Matrix')
cm = confusion_matrix(test_generator.classes, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=test_generator.class_indices.keys(), yticklabels=test_generator.class_indices.keys())
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.show()

print('Classification Report')
print(classification_report(test_generator.classes, y_pred, target_names=test_generator.class_indices.keys()))